# PatchTST-Only Optuna Tuning (HF, Kaggle, 2-GPU)

This notebook covers the end-to-end tuning workflow for the PatchTST classifier:
- downloads the base and VIIRS split files from Hugging Face
- merges them on the shared spatial/temporal keys
- normalizes features using train statistics only
- runs Optuna over the PatchTST search space
- stores the best study result for the final training pass

The notebook is organized into short, rerunnable stages so each step can be debugged independently.


## Environment Setup

Install the small set of Python packages that are not guaranteed to be present in a fresh Kaggle session. This cell is safe to rerun.


In [1]:
# Kaggle installs (safe to re-run)
!pip -q install optuna shap huggingface_hub


## Imports and Shared Helpers

Load the standard library, numerical stack, plotting tools, PyTorch, Optuna, SHAP, and Hugging Face download helpers used throughout the notebook.


In [2]:
from pathlib import Path
import os
import sys
import json
import copy
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import optuna
import shap

from huggingface_hub import hf_hub_download

## Resolve the Model Package

The tuning notebook reuses the shared `MODEL/src` codebase. This cell locates the package, adds it to `sys.path`, and imports the training and data utilities used by the study.


In [3]:
# -------------------------
# Resolve MODEL/src package
# -------------------------
SRC_CANDIDATES = [
    Path('/kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1')
]

MODEL_SRC = None
for cand in SRC_CANDIDATES:
    if cand.exists():
        MODEL_SRC = cand
        break
if MODEL_SRC is None:
    raise FileNotFoundError(f'Could not find MODEL/src from: {SRC_CANDIDATES}')

SRC_PARENT = MODEL_SRC.parent if MODEL_SRC.name == 'src' else MODEL_SRC
if str(SRC_PARENT) not in sys.path:
    sys.path.insert(0, str(SRC_PARENT))

from src import (
    DataConfig,
    ModelConfig,
    TrainConfig,
    PatchTSTDLAClassifier,
    load_split_xy_csv,
    FeatureNormalizer,
    make_torch_datasets,
    make_dataloaders,
    fit_model,
    evaluate_with_threshold_search,
)
from src.utils import set_seed, device_summary

print('MODEL_SRC:', MODEL_SRC)
print('GPU:', device_summary())


MODEL_SRC: /kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1
GPU: {'cuda_available': True, 'n_gpu': 2, 'gpu_names': ['Tesla T4', 'Tesla T4']}


## Runtime Configuration

Set seeds, choose the Hugging Face repositories, define the Optuna budget, and load the Kaggle secret token if available. The two-GPU check happens here so the notebook fails fast on the wrong hardware.


In [4]:
# -------------------------
# Config
# -------------------------
SEED = 42
set_seed(SEED, deterministic=True)
np.random.seed(SEED)
random.seed(SEED)

# HF repos
HF_X_REPO = 'NagrajMG/WildFire-X'  # has both base and *_viirs files
HF_Y_REPO = 'NagrajMG/WildFire-Y'

HF_X_BASE_FILE_TEMPLATE = 'FEATURES_{split}.csv'
HF_X_VIIRS_FILE_TEMPLATE = 'FEATURES_{split}_viirs.csv'
HF_Y_FILE_TEMPLATE = 'LABELS_{split}.csv'

USE_VIIRS_FEATURES = True
EXPECTED_VIIRS_FEATURES = 18
STRICT_VIIRS_KEY_MATCH = True

# GPU policy
USE_MULTI_GPU = True
REQUIRE_TWO_GPUS = True

# Optuna
N_TRIALS = 80
OPTUNA_METRIC = 'pr_auc'   # pr_auc | f1 | iou
OPTUNA_DIRECTION = 'maximize'
STUDY_NAME = 'Final_Optuna_Run'
STORAGE = 'sqlite:///optuna_full_space_viirs_v1.db'

# Train budget
TUNE_EPOCHS = 22
FINAL_EPOCHS = 36
PATIENCE = 5

# DataLoader defaults
BASE_NUM_WORKERS = 4
BASE_PIN_MEMORY = True

WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/tmp')
ARTIFACT_DIR = WORK_DIR / 'optuna_full_space_viirs'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# HF token (Kaggle secret first)
HF_TOKEN = os.getenv('HF_TOKEN', None)
try:
    from kaggle_secrets import UserSecretsClient
    ksc = UserSecretsClient()
    for key in ['HUGGINGFACE_TOKEN', 'HF_TOKEN']:
        try:
            tok = ksc.get_secret(key)
            if tok:
                HF_TOKEN = tok
                break
        except Exception:
            pass
except Exception:
    pass

if REQUIRE_TWO_GPUS:
    n_gpu = torch.cuda.device_count() if torch.cuda.is_available() else 0
    if n_gpu < 2:
        raise RuntimeError(f'REQUIRE_TWO_GPUS=True but only {n_gpu} GPU(s) available')

print('Artifacts:', ARTIFACT_DIR)
print('GPU summary:', device_summary())
print('HF token present:', bool(HF_TOKEN))




Artifacts: /kaggle/working/optuna_full_space_viirs
GPU summary: {'cuda_available': True, 'n_gpu': 2, 'gpu_names': ['Tesla T4', 'Tesla T4']}
HF token present: True


## Fetch Split Files

Download the train, validation, and test feature tables and labels from Hugging Face into the local Kaggle cache. Keeping the download step isolated makes it easier to retry without rerunning the full notebook.


In [5]:
# -------------------------
# Download split files from HF
# -------------------------
SPLITS = ['train', 'val', 'test']
paths = {}
for split in SPLITS:
    paths[f'x_base_{split}'] = hf_hub_download(
        repo_id=HF_X_REPO,
        filename=HF_X_BASE_FILE_TEMPLATE.format(split=split),
        repo_type='dataset',
        token=HF_TOKEN,
        cache_dir=WORK_DIR / 'hf_cache',
    )
    if USE_VIIRS_FEATURES:
        paths[f'x_viirs_{split}'] = hf_hub_download(
            repo_id=HF_X_REPO,
            filename=HF_X_VIIRS_FILE_TEMPLATE.format(split=split),
            repo_type='dataset',
            token=HF_TOKEN,
            cache_dir=WORK_DIR / 'hf_cache',
        )
    paths[f'y_{split}'] = hf_hub_download(
        repo_id=HF_Y_REPO,
        filename=HF_Y_FILE_TEMPLATE.format(split=split),
        repo_type='dataset',
        token=HF_TOKEN,
        cache_dir=WORK_DIR / 'hf_cache',
    )

print(json.dumps(paths, indent=2))


{
  "x_base_train": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-X/snapshots/6c3fd5e077deaa7afeb4b8ae10860505a3440633/FEATURES_train.csv",
  "x_viirs_train": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-X/snapshots/6c3fd5e077deaa7afeb4b8ae10860505a3440633/FEATURES_train_viirs.csv",
  "y_train": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-Y/snapshots/5c10f82574a79f7dc8939387f7e39e1c275ba0dc/LABELS_train.csv",
  "x_base_val": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-X/snapshots/6c3fd5e077deaa7afeb4b8ae10860505a3440633/FEATURES_val.csv",
  "x_viirs_val": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-X/snapshots/6c3fd5e077deaa7afeb4b8ae10860505a3440633/FEATURES_val_viirs.csv",
  "y_val": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-Y/snapshots/5c10f82574a79f7dc8939387f7e39e1c275ba0dc/LABELS_val.csv",
  "x_base_test": "/kaggle/working/hf_cache/datasets--NagrajMG--WildFire-X/snapshots/6c3fd5e077deaa7afeb4b8ae10860505a3440633/FEA

## Merge Feature Tables

Join the base feature CSVs with the VIIRS feature CSVs on the shared spatial-temporal key columns. The merge is strict by default so key mismatches surface immediately.


In [6]:
# -------------------------
# Merge base + VIIRS feature CSVs per split
# -------------------------
KEY_COLS = ['target_date', 'row', 'col', 'window_id']


def merge_feature_csvs(base_csv, viirs_csv, out_csv, split_name, strict_match=True):
    base_df = pd.read_csv(base_csv).drop_duplicates(KEY_COLS, keep='first')
    viirs_df = pd.read_csv(viirs_csv).drop_duplicates(KEY_COLS, keep='first')

    for c in KEY_COLS:
        if c not in base_df.columns:
            raise ValueError(f'{base_csv} missing key: {c}')
        if c not in viirs_df.columns:
            raise ValueError(f'{viirs_csv} missing key: {c}')

    viirs_cols = [c for c in viirs_df.columns if c not in KEY_COLS and c != 'split']
    if not viirs_cols:
        raise ValueError(f'No VIIRS feature columns in {viirs_csv}')

    base_cols = [c for c in base_df.columns if c not in KEY_COLS and c != 'split']
    collisions = sorted(set(base_cols).intersection(viirs_cols))
    if collisions:
        rename = {c: f'viirs_{c}' for c in collisions}
        viirs_df = viirs_df.rename(columns=rename)
        viirs_cols = [rename.get(c, c) for c in viirs_cols]

    merged = base_df.merge(viirs_df[KEY_COLS + viirs_cols], on=KEY_COLS, how='left', validate='one_to_one')

    all_missing = merged[viirs_cols].isna().all(axis=1)
    n_missing = int(all_missing.sum())
    if strict_match and n_missing > 0:
        raise ValueError(f'split={split_name}: {n_missing} keys have no VIIRS match')

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(out_csv, index=False)

    return {
        'rows_base': int(len(base_df)),
        'rows_viirs': int(len(viirs_df)),
        'rows_merged': int(len(merged)),
        'n_viirs_features_added': int(len(viirs_cols)),
        'n_missing_viirs_keys': int(n_missing),
        'viirs_cols': viirs_cols,
    }


merged_dir = ARTIFACT_DIR / 'merged_features'
merged_dir.mkdir(parents=True, exist_ok=True)
merge_stats = {}

for split in SPLITS:
    if USE_VIIRS_FEATURES:
        merged_path = merged_dir / f'FEATURES_{split}_with_viirs.csv'
        st = merge_feature_csvs(
            base_csv=paths[f'x_base_{split}'],
            viirs_csv=paths[f'x_viirs_{split}'],
            out_csv=merged_path,
            split_name=split,
            strict_match=STRICT_VIIRS_KEY_MATCH,
        )
        paths[f'x_{split}'] = str(merged_path)
        merge_stats[split] = st
    else:
        paths[f'x_{split}'] = paths[f'x_base_{split}']

if USE_VIIRS_FEATURES:
    print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'viirs_cols'} for k, v in merge_stats.items()}, indent=2))
    n_added = int(merge_stats['train']['n_viirs_features_added'])
    if n_added != int(EXPECTED_VIIRS_FEATURES):
        print(f'Warning: expected {EXPECTED_VIIRS_FEATURES} VIIRS features, found {n_added}')


{
  "train": {
    "rows_base": 190120,
    "rows_viirs": 190120,
    "rows_merged": 190120,
    "n_viirs_features_added": 18,
    "n_missing_viirs_keys": 0
  },
  "val": {
    "rows_base": 41160,
    "rows_viirs": 41160,
    "rows_merged": 41160,
    "n_viirs_features_added": 18,
    "n_missing_viirs_keys": 0
  },
  "test": {
    "rows_base": 7560,
    "rows_viirs": 7560,
    "rows_merged": 7560,
    "n_viirs_features_added": 18,
    "n_missing_viirs_keys": 0
  }
}


## Build Arrays and Normalize

Load each split into model-ready arrays, then fit the feature normalizer on train only to avoid leakage into validation or test. The dataset objects are kept stable so later trials reuse the same in-memory objects.


In [7]:
# -------------------------
# Load arrays and normalize (train-only)
# -------------------------
base_data_cfg = DataConfig(
    expected_windows=4,
    batch_size=512,
    num_workers=BASE_NUM_WORKERS,
    pin_memory=BASE_PIN_MEMORY,
    drop_last_train=False,
)

train_arr = load_split_xy_csv(paths['x_train'], paths['y_train'], split_name='train', expected_windows=base_data_cfg.expected_windows)
val_arr   = load_split_xy_csv(paths['x_val'],   paths['y_val'],   split_name='val',   expected_windows=base_data_cfg.expected_windows)
test_arr  = load_split_xy_csv(paths['x_test'],  paths['y_test'],  split_name='test',  expected_windows=base_data_cfg.expected_windows)

print('before norm train/val/test:', train_arr.x.shape, val_arr.x.shape, test_arr.x.shape)
print('feature count:', len(train_arr.feature_cols))

normalizer = FeatureNormalizer()
train_arr.x = normalizer.fit_transform(train_arr.x)
val_arr.x   = normalizer.transform(val_arr.x)
test_arr.x  = normalizer.transform(test_arr.x)

# keep dataset objects stable across trials
train_ds, val_ds, test_ds = make_torch_datasets(train_arr, val_arr, test_arr)

print('after norm train mean/std (rough):', float(train_arr.x.mean()), float(train_arr.x.std()))


before norm train/val/test: (47530, 4, 43) (10290, 4, 43) (1890, 4, 43)
feature count: 43
after norm train mean/std (rough): -5.054433316331597e-09 1.000000238418579


## Define the Search Space

Sample the PatchTST architecture, fusion branches, and optimization settings from the full trial space. The helper functions here translate Optuna parameters into the model, data, and training configs used by each run.


In [8]:
# -------------------------
# Build trial configs (FULL search space)
# -------------------------
VALID_HEAD_CHOICES = ['64x4', '96x6', '128x8', '160x10', '192x12', '256x16']


def _sample_trial_params(trial):
    p = {}
    p['dmodel_heads'] = trial.suggest_categorical('dmodel_heads', VALID_HEAD_CHOICES)

    p['patch_len'] = trial.suggest_categorical('patch_len', [2, 4])
    if p['patch_len'] == 4:
        p['patch_stride'] = trial.suggest_categorical('patch_stride', [1, 2])
    else:
        p['patch_stride'] = 1

    p['n_layers'] = trial.suggest_int('n_layers', 2, 8)
    p['ff_mult'] = trial.suggest_categorical('ff_mult', [2, 4, 6])
    p['dropout'] = trial.suggest_float('dropout', 0.05, 0.35)
    p['attn_dropout'] = trial.suggest_float('attn_dropout', 0.00, 0.30)

    p['mla_hidden'] = trial.suggest_categorical('mla_hidden', [64, 128, 192, 256])
    p['head_hidden'] = trial.suggest_categorical('head_hidden', [64, 128, 192, 256])

    # Full-model toggles (no patchtst-only restriction)
    p['use_temporal_dla1d'] = trial.suggest_categorical('use_temporal_dla1d', [True, False])
    p['temporal_dla_base_channels'] = trial.suggest_categorical('temporal_dla_base_channels', [16, 32, 48, 64])
    p['use_temporal_attention_pool'] = trial.suggest_categorical('use_temporal_attention_pool', [True, False])
    p['use_window_embedding'] = trial.suggest_categorical('use_window_embedding', [True, False])
    p['use_summary_branches'] = trial.suggest_categorical('use_summary_branches', [True, False])
    p['use_gated_fusion'] = trial.suggest_categorical('use_gated_fusion', [True, False])
    p['fusion_hidden'] = trial.suggest_categorical('fusion_hidden', [64, 128, 192, 256])

    p['use_feature_mixer'] = trial.suggest_categorical('use_feature_mixer', [True, False])
    p['feature_mixer_hidden_mult'] = trial.suggest_categorical('feature_mixer_hidden_mult', [1, 2, 3, 4])
    p['feature_mixer_dropout'] = trial.suggest_categorical('feature_mixer_dropout', [0.0, 0.05, 0.10, 0.20])

    p['use_tabular_shortcut'] = trial.suggest_categorical('use_tabular_shortcut', [True, False])
    p['tabular_hidden'] = trial.suggest_categorical('tabular_hidden', [128, 256, 384])

    p['batch_size'] = trial.suggest_categorical('batch_size', [256, 512, 1024, 1536])
    p['lr'] = trial.suggest_float('lr', 5e-5, 5e-3, log=True)
    p['weight_decay'] = trial.suggest_float('weight_decay', 1e-7, 1e-2, log=True)
    p['grad_clip_norm'] = trial.suggest_categorical('grad_clip_norm', [0.0, 0.5, 1.0, 2.0, 3.0])
    p['patience'] = trial.suggest_categorical('patience', [5, 6, 8, 10])

    p['loss_name'] = trial.suggest_categorical('loss_name', ['bce', 'focal'])
    p['pos_weight'] = trial.suggest_categorical('pos_weight', [1.0, 1.5, 2.0])
    p['focal_gamma'] = trial.suggest_categorical('focal_gamma', [1.5, 2.0, 2.5])

    p['scheduler_name'] = trial.suggest_categorical('scheduler_name', ['none', 'cosine'])
    p['min_lr'] = trial.suggest_categorical('min_lr', [1e-6, 1e-5, 5e-5])
    return p


def build_configs_from_params(params: dict, n_features: int, epochs: int):
    d_model, n_heads = [int(x) for x in str(params['dmodel_heads']).split('x')]

    model_cfg = ModelConfig(
        seq_len=4,
        patch_len=int(params['patch_len']),
        patch_stride=int(params['patch_stride']),
        d_model=int(d_model),
        n_heads=int(n_heads),
        n_layers=int(params['n_layers']),
        ff_mult=int(params['ff_mult']),
        dropout=float(params['dropout']),
        attn_dropout=float(params['attn_dropout']),
        n_features=int(n_features),

        mla_hidden=int(params['mla_hidden']),
        head_hidden=int(params['head_hidden']),

        use_temporal_dla1d=bool(params['use_temporal_dla1d']),
        temporal_dla_base_channels=int(params['temporal_dla_base_channels']),
        use_temporal_attention_pool=bool(params['use_temporal_attention_pool']),
        use_window_embedding=bool(params['use_window_embedding']),
        use_summary_branches=bool(params['use_summary_branches']),
        use_gated_fusion=bool(params['use_gated_fusion']),
        fusion_hidden=int(params['fusion_hidden']),

        use_feature_mixer=bool(params['use_feature_mixer']),
        feature_mixer_hidden_mult=int(params['feature_mixer_hidden_mult']),
        feature_mixer_dropout=float(params['feature_mixer_dropout']),

        use_tabular_shortcut=bool(params['use_tabular_shortcut']),
        tabular_hidden=int(params['tabular_hidden']),

        patchtst_only=False,
    )

    data_cfg = DataConfig(
        expected_windows=4,
        batch_size=int(params['batch_size']),
        num_workers=BASE_NUM_WORKERS,
        pin_memory=BASE_PIN_MEMORY,
        drop_last_train=False,
    )

    train_cfg = TrainConfig(
        epochs=int(epochs),
        lr=float(params['lr']),
        weight_decay=float(params['weight_decay']),
        grad_clip_norm=float(params['grad_clip_norm']),
        patience=int(params['patience']),
        use_amp=True,
        loss_name=str(params['loss_name']),
        pos_weight=float(params['pos_weight']),
        focal_gamma=float(params['focal_gamma']),
        monitor_metric=str(OPTUNA_METRIC),
        scheduler_name=str(params['scheduler_name']),
        min_lr=float(params['min_lr']),
    )

    return model_cfg, train_cfg, data_cfg


def build_trial_configs(trial, n_features: int):
    params = _sample_trial_params(trial)
    return build_configs_from_params(params, n_features=n_features, epochs=int(TUNE_EPOCHS))



## Optuna Objective

Build a trial-specific configuration, train for the tuning budget, and return the monitored validation score. OOMs and invalid scores are treated as prunable trials so the study can continue cleanly.


In [9]:
# -------------------------
# Optuna objective
# -------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def objective(trial):
    set_seed(SEED + int(trial.number), deterministic=True)

    model_cfg, train_cfg, data_cfg = build_trial_configs(trial, n_features=train_arr.x.shape[2])
    trial.set_user_attr('n_features', int(train_arr.x.shape[2]))
    trial.set_user_attr('device', str(device))

    train_loader, val_loader, _ = make_dataloaders(train_ds, val_ds, test_ds, data_cfg)
    model = PatchTSTDLAClassifier(model_cfg)

    try:
        out = fit_model(
            model=model,
            train_loader=train_loader,
            val_loader=val_loader,
            cfg=train_cfg,
            device=device,
            use_multi_gpu=USE_MULTI_GPU,
        )
        score = float(out.get('best_monitor_value', np.nan))
        if not np.isfinite(score):
            raise optuna.TrialPruned('non-finite score')
        return score
    except RuntimeError as e:
        if 'out of memory' in str(e).lower():
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            trial.set_user_attr('oom', True)
            raise optuna.TrialPruned('CUDA OOM')
        raise


## Run the Study

Create or resume the persistent Optuna study, execute the requested number of trials, and print the best trial summary for the final training pass.


In [ ]:
# -------------------------
# Run study
# -------------------------
sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    study_name=STUDY_NAME,
    direction=OPTUNA_DIRECTION,
    sampler=sampler,
    storage=STORAGE,
    load_if_exists=True,
)

study.optimize(objective, n_trials=int(N_TRIALS), gc_after_trial=True)

print('Best trial:', study.best_trial.number)
print('Best value:', study.best_value)
print('Best params:', study.best_trial.params)


[I 2026-04-30 17:36:17,031] A new study created in RDB with name: Final_Optuna_Run
/kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1/src/trainer.py:204: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=bool(cfg.use_amp and device.type == "cuda"))
/kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1/src/trainer.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
/kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1/src/trainer.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=amp_enabled):
/kaggle/input/models/nagrajgaonkar/nagrajmg/pytorch/cv5600/1/src/trainer.py:122: FutureWarning: `torch.cu